## Aim:
How to link the rule-based extracted CI_TYPE-GEO pairs with the respective Ci failure impacts

**Idea**\
Test using prompt engineering by passing table of CIGEO pairs to GPT-J model.
Steps:
* Load model and apply it always on one chunk of the document to extract CI failure impacts 
* Use prompt engineering to extract time and location of the CI failure (origin) the CI impacts (impact location)
or 
* Pass dataframe of pairs as input to the model
or
* Use few shot prompting with example answers

**Finally:**
* Compaire all approaches of spatial and temporal linking CI failure impacts

In [1]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0  # nvidia gpu
%env PYTORCH_ALLOC_CONF=expandable_segments:True
# %env TORCH_CUDA_ARCH_LIST=8.6

# settings for distributed computing
%env WORLD_SIZE=1
%env RANK=0
%env LOCAL_RANK=0

# NOTE: # WORLD_SIZE: each GPU corresponds to one process (world = no. of processes within a group), processes communicate with each other enabling eg., distributed training
# NOTE: # RANK: IDs of the processes, ranging from 0 up to WORLD_SIZE - 1

env: CUDA_DEVICE_ORDER=PCI_BUS_ID
env: CUDA_VISIBLE_DEVICES=0  # nvidia gpu
env: PYTORCH_ALLOC_CONF=expandable_segments:True
env: WORLD_SIZE=1
env: RANK=0
env: LOCAL_RANK=0


In [10]:
import os
import sys
import re
from glob import glob
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
from jinja2 import Template
import spacy
import langextract as lx
import textwrap
from langchain_docling import DoclingLoader
from huggingface_hub import login
import torch


sys.path.append("../")
from src.settings import settings as s

torch.manual_seed(42)

# set default location to store model before loading transformers
os.environ["HF_HOME"] = (
    "/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/"
)



In [6]:
s.SPACY_MODEL

'en_core_web_trf'

## Generate CI_GEO-pairs

In [7]:
# import spacy_transformers


try:
    nlp = spacy.load(s.SPACY_MODEL)
except (OSError, ValueError):
    print(f"spaCy language model '{s.SPACY_MODEL}' not found. Downloading ...")
    ## loading transformer language model for NER requires additional package
    if (
        s.SPACY_MODEL.endswith("_trf")
        and importlib.util.find_spec("spacy[transformers]") is None
    ):
        !uv add spacy[transformers]
    !uv run python -m spacy download {s.SPACY_MODEL}
    nlp = spacy.load(s.SPACY_MODEL)

print(f"Loaded spaCy language model: {s.SPACY_MODEL}")

Loaded spaCy language model: en_core_web_trf


In [16]:
## Create New entity for transport infrastructure and apply it on any doc

## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()


## call nlp model and create pipeline with new entity pattern
# NOTE Creating new entity (CI_TYPE) solves the issue that FAC entities (buildings, airports, highways, bridges, etc.) refer only to the name of the facility (e.g. A76, Ahrtalbahn)
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk("../ner_patterns.jsonl")
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk("../ner_patterns.jsonl")


# store patterns in jsonl file, example:
# ruler.add_patterns([
#     {"label": "CI_TYPE", "pattern": "road?.+"},
#    {"label":"CI_TYPE","pattern":"rail.*$"},
# ])
# ruler.to_disk("../ner_patterns.jsonl")


## load docs
PARSED_TEXT_DIR = "../" + s.PATH_DATA + "parsed_documents/"
docs_list = glob(PARSED_TEXT_DIR + "*_cleaned.md")
print(f"Found {len(docs_list)} cleaned documents.")


## DataFrame to store CI-GEO entity pairs
df_ci_geo = pd.DataFrame(     ## TODO make as pydantic class with fixed attributes
    columns=[
        "document_id",
        "chunk_id",
        "ci_entity",
        "ci_entity_label",
        "geo_entity",
        "geo_entity_label",
        "token_distance",
    ]
)


## iterate over all cleaned documents and extract CI-GEO entity pairs

for FILE_PATH in docs_list:
    print(f"\n\n Loading  - {Path(FILE_PATH).name} - ")
    loader = DoclingLoader(FILE_PATH)  # use chunks from Docling.Loader
    doc = loader.load()
  

    ## get most likely geolocation for each CI entity based on distance
    for i, chunk in enumerate(doc):
        nlp_chunk = nlp(chunk.page_content)
        all_ents = [ent for ent in nlp_chunk.ents]
        ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]
        ci_type_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE"]]
        fac_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["FAC"]]

        # check if chunk contains CI_TYPE entities
        if len(ci_type_ents) > 0:
            print(f"\nChunk [{i}], No. CI_TYPE and FAC entities: {len(ci_type_ents)}")
            print(
                f"Contains following entities for CI_TYPE: {ci_type_ents_info}, FAC: {fac_ents_info}"
            )
            print(f"Chunk text [{i}]:", chunk.page_content)
            # print(f"{ {(ci_type_ents[i].text, ci_type_ents[i].label_) for i in range(len(ci_type_ents))} } ")

            # iterate over all entities within chunk
            for ent_idx in range(len(all_ents)):
                # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
                if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
                    ci_idx = ent_idx

                    ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
                    distance_list = []
                    idx_in_chunk = []
                    try:
                        for ent_idx in range(len(all_ents)):
                            # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities
                            if all_ents[ent_idx].label_ in ["GPE", "LOC"]:
                                geo_idx = ent_idx
                                dist_ent_pair = np.abs(ci_idx - geo_idx)
                                distance_list.append(dist_ent_pair)
                                idx_in_chunk.append((ent_idx))
                                closest_pair_idx = np.argmin(
                                    distance_list
                                )  # idx of closest GEO entity
                                distance_closest_pair = distance_list[closest_pair_idx]

                        threshold = 5  # max token distance between CI_TYPE and GEO entity
                        if distance_closest_pair > threshold:
                            print(
                                f""" Token distance between CI_TYPE/FAR and next GEO entity is {distance_closest_pair} and thus larger than the allowed distance of {threshold} tokens """
                            )
                            continue
                        else:
                            print(
                                f"""  Closest GEO entity to CI_TYPE/FAC entity "{all_ents[ci_idx]}" is "{all_ents[idx_in_chunk[closest_pair_idx]]}" at distance {distance_closest_pair}"""
                            )  # TODO constrain min.distance to max value (eg. 5 tokens), issue: likely when distance value is high that geolocation of Ci_type is mentioned in previous sentences or chunk

                        ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
                        result_dict = {
                            "document_id": Path(FILE_PATH).stem,
                            "chunk_id": i,
                            "ci_entity": all_ents[ci_idx].text,
                            "ci_entity_label": all_ents[ci_idx].label_,
                            "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
                            "geo_entity_label": all_ents[
                                idx_in_chunk[closest_pair_idx]
                            ].label_,
                            "token_distance": distance_closest_pair,
                        }
                        df_ci_geo = pd.concat(
                            [df_ci_geo, pd.DataFrame([result_dict])], ignore_index=True
                        )

                    except IndexError:
                        print("No GEO entities found in this chunk.")
                        continue
                    # print("\nidx_in_chunk, closest pair idx", idx_in_chunk, closest_pair_idx)

                    # spacy.displacy.render(
                    #     nlp_chunk, style="ent",
                    #     options={"ents": ["CI_TYPE", "GPE", "LOC", "FAC"], "colors": {"CI_TYPE": "violet"}}
                    # )

            else:
                print("\nNo CI_TYPE or FAC entities found in this chunk.")
                continue

Found 32 cleaned documents.


 Loading  - EFE 2024 - The DANA storm, live_ The death toll rises to 158_cleaned.md - 


2026-01-06 22:02:31,520 - INFO - Going to convert document batch...
2026-01-06 22:02:31,520 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2026-01-06 22:02:31,521 - INFO - Processing document EFE 2024 - The DANA storm, live_ The death toll rises to 158_cleaned.md
2026-01-06 22:02:31,676 - INFO - Finished converting document EFE 2024 - The DANA storm, live_ The death toll rises to 158_cleaned.md in 0.16 sec.



Chunk [0], No. CI_TYPE and FAC entities: 1
Contains following entities for CI_TYPE: [emergency], FAC: []
Chunk text [0]: rises to 158 and Mazón requests the Army's help in Valencia
Valencia (EFE) - The provisional death toll from the floods in Spain has risen to
158, of which 155 have died in the province of Valencia, another 2 in Castilla-La Mancha, and 1 more in Andalusia. However, an undetermined number of people
remain missing, and the emergency is not yet over.
https://efe.com/espana/2024-10-31/dana-directo-ultima-hora/
Several people work on the cleanup and debris removal in Paiporta, Valencia. EFE/Biel Aliño1/3/26, 11:28 PM
The latest count provided by the Valencian Regional Government's Emergency
Services has raised the number of fatalities to 155 in the last few hours, and this number may increase as intervention and rescue teams gain access to the
One of the hardest-hit towns has been Paiporta, just ten kilometers from the city of
Valencia, where the remains of some 45 peop

2026-01-06 22:02:37,535 - INFO - Going to convert document batch...
2026-01-06 22:02:37,535 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2026-01-06 22:02:37,535 - INFO - Processing document Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News_cleaned.md
2026-01-06 22:02:37,543 - INFO - Finished converting document Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News_cleaned.md in 0.01 sec.



Chunk [1], No. CI_TYPE and FAC entities: 2
Contains following entities for CI_TYPE: [rail, rail], FAC: []
Chunk text [1]: BARCELONA, Spain (AP) — Several people were reported missing by Spanish authorities after ﬂash ﬂoodsswept cars through village streets and disrupted rail service in large areas of eastern and southern Spain onTuesday.Rushing mud-colored waters caused havoc in a huge arc of the European country, running from the provincesof Malaga in the south to Valencia in the east. Images shot by people with smartphones reproduced onSpain’s national broadcaster RTVE showed frighteningly swift waters carrying away cars and rising severalfeet into the lower level of homes.A high-speed train with nearly 300 people on board derailed near Malaga, although rail authorities said noone was hurt. The high-speed train service between Valencia city and Madrid was interrupted as were severalcommuter lines.The national government ofﬁce for the Castilla La Mancha region told radio channel Cade

...

2026-01-06 22:06:06,650 - INFO - Going to convert document batch...
2026-01-06 22:06:06,651 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2026-01-06 22:06:06,652 - INFO - Processing document The Guardian 2018 - Freezing weather costs UK economy £1bn a day _ UK weather_cleaned.md
2026-01-06 22:06:06,714 - INFO - Finished converting document The Guardian 2018 - Freezing weather costs UK economy £1bn a day _ UK weather_cleaned.md in 0.07 sec.



Chunk [0], No. CI_TYPE and FAC entities: 4
Contains following entities for CI_TYPE: [motorways, rail, motorways, roads], FAC: []
Chunk text [0]: Financial impact of the ‘beast from the east’ and storm Emma worst since Christmas 2010
Gridlocked motorways, empty restaurants and idle diggers seen across Britain last week cost the economy at least £1bn a day and could halve GDP growth in the ﬁrst three months of the year.
Analysts said the impact of the “beast from the east” sweeping in from Siberia and the arrival of Storm Emma hitting the south coast was likely to be the most costly weather event since 2010, when freezing temperatures and snow brought the economy to a standstill a week before Christmas.
The extreme weather was likely to have the biggest impact on the construction industry, which experts said could lose up to £2bn over the three worst days, as sub-zero temperatures forced building workers to down tools.
Transport networks and retailers were also expected to count the cos

2026-01-06 22:06:09,337 - INFO - Going to convert document batch...
2026-01-06 22:06:09,338 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2026-01-06 22:06:09,339 - INFO - Processing document Karakatsani 2023 - Greece economy briefing The economic impact of the recent devastating floods in Greece_cleaned.md
2026-01-06 22:06:09,440 - INFO - Finished converting document Karakatsani 2023 - Greece economy briefing The economic impact of the recent devastating floods in Greece_cleaned.md in 0.10 sec.



Chunk [0], No. CI_TYPE and FAC entities: 4
Contains following entities for CI_TYPE: [roads, bridges, railway], FAC: [National]
Chunk text [0]: The economic impact of the recent devastating floods in Greece 
The economic impact of the recent devastating floods in Greece 
The briefing presents the economic impact of the damages caused by the storm “Daniel”. 
The  floods  caused  by  the  heavy  rainfall,  especially  in  Thessaly  plain  -accounting  for 
approximately  15%  of  Greece’s  agricultural  land-  resulted  to  a  massive  destruction  in 
agriculture,  infrastructure  and  residences,  which  is  expected  to  negatively  affect  the  Greek 
economy in short and mid-term. Fears for food shortages and increase of product prices have 
been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the 
economy in the upcoming years. The extent of the impact to the economy is not yet know. 
Before the country recovered from the wildfires of August 

2026-01-06 22:06:12,927 - INFO - Going to convert document batch...
2026-01-06 22:06:12,928 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2026-01-06 22:06:12,929 - INFO - Processing document European Investment Bank 2025 - Spain_ EIB lends €50 million to Iberdrola to rebuild and climate-proof flood-hit power infrastructure in Valencia_cleaned.md
2026-01-06 22:06:12,974 - INFO - Finished converting document European Investment Bank 2025 - Spain_ EIB lends €50 million to Iberdrola to rebuild and climate-proof flood-hit power infrastructure in Valencia_cleaned.md in 0.06 sec.


In [ ]:
df_ci_geo.loc[df_ci_geo["ci_entity"] == "rail"]  # .head(15)

In [ ]:
doc[13].page_content

## LLama with LangExtract

###  Prompt engineering

In [ ]:
question = "Which impacts of infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, societal or economic impacts, the location and possibly the time of the infrastructure failure."

## without s+e impacts
# question = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location and possibly the time of the infrastructure failure."

In [ ]:
# TODO move template to separate file and load via get_template(), define conditions (e.g. user is technical or not)
# TODo make pydantic class model for expected JSON output

# Example code: https://medium.com/@alecgg27895/jinja2-prompting-a-guide-on-using-jinja2-templates-for-prompt-management-in-genai-applications-e36e5c1243cf
# test instead of user_type (see: {% block user_type %}) the modification of question in regard to CI impact types (Tier 1,2,3 and 4 )


## first
#     You are an expert analyst assistant and should use ONLY the provided context to answer the following question:

## Text_snippet column
#     In the field "text_snippet" provide the exact linenumbers (within a list) from the context which you used to extract the information about the infrastructure failure and its impacts.
## too long responses


# prompt_template = """

#     You are an expert analyst assistant and should use ONLY the provided context to answer the following question:

#     Question: "{{ question }}"

#     Context:
#     {% for item in context %}
#     - {{ item.text }} (Citation: {{ item.citation }})
#     {% endfor %}

#     Evaluate and improve your answer based on the information about critical infrastructure (CI) types (column: "ci_entity") and their geolocations (column: "geo_entity") mentioned in CI locations.

#     CI locations:
#     {% for item in context %}
#     - ("ci_entity" and "geo_entity":\n {{ item["ci_locations"][["ci_entity", "geo_entity"]] }})
#     {% endfor %}


#     In the field "confidence" you should give an estimate how confident you are about the provided information on a scale between 1 (low) to 5 (high).
#     In the "confidence_explanation" field give also a short explanation why you decided for a certain confidence value (maximum two bullet points within a list).


#     Return ONLY valid JSON in the following list format:
#     [{
#         "infrastructure_type": "...",
#         "damage": "...",
#         "location": "...",
#         "time": "...",
#         "duration": "...",
#         "confidence": "...",
#         "confidence_explanation": "[...]"
#     }]

#     Each nested dictionary describes one failure case.
#     DO NOT add commentary or text outside the JSON.


#     Answer:
# """

######################################################################


prompt_template = """

    You are an expert analyst assistant and should use ONLY the provided context to answer the following question:
    
    Question: "{{ question }}"
       
    Context:
    {% for item in context %}
    - {{ item.text }} (Citation: {{ item.citation }})
    {% endfor %}

    
    For the the fields "societal_impact" and "economic_impact" you should try to extract information about societal or economic consequences of infrastructure failures mentioned in the context.
    However, if you do not find any information about societal or economic consequences, then return for these fields a "NAN" value.
    
    For the fields ""infrastructure_type" and "location" you should evaluate and improve your answer based on the information mentioned in CI locations.

    CI locations:
    {% for item in context %}
    - ("ci_entity" and "geo_entity":\n {{ item["ci_locations"][["ci_entity", "geo_entity"]] }})
    {% endfor %}


    In the field "confidence" you should give an estimate how confident you are about the provided information in regard to the societal and economic impacts on a scale between 1 (low) to 5 (high). 
    In the "confidence_explanation" field give also a very short explanation why you decided for a certain confidence value (maximum two bullet points within a list).


    Return ONLY valid JSON in the following list format:
    [{
        "infrastructure_type": "...",
        "damage": "...",
        "societal_impact": "...",
        "economic_impact": "...",
        "location": "...",
        "confidence_explanation": "[...]",
        "confidence": "...",
    }]

    Each nested dictionary describes one failure case.
    DO NOT add commentary or text outside the JSON.
    

    Answer:
"""

template = Template(prompt_template)


# example_context = doc[6].page_content
# chunk_id = 6
# context = [
#     {
#         "text": example_context,
#         "citation": "Koks et al., 2022",
#         "ci_locations": df_ci_geo.loc[df_ci_geo["chunk_id"]==chunk_id],#to_dict(orient="records")
#     },  # TODO use author names or Primary keys from DB
#     # {"text": context, "citation": "Meier et al., 2025"},
# ]

# rendered_prompt = template.render(
#     context=context,
#     question=question,
#     # messages=messages
# )
# print(rendered_prompt)


## left overs
#  Try to be as specific as possible in your answer (bullet points), mention the impacts as numerical information along the location of the impact, and refer to the citations provided in the context.
# # Extract information about infrastructure failures based on the following question:

##  Test Gemini with LangExtract
Langextract does not support Meta models (eg llama) directly (only via Ollama). For this reason we use an alternative for now with an easier implementation: Googles gemini\

**Local models with Ollama**\
see later section 


Optional: if you use API-based packages eg. fastchat and vLLM for passing HF models to LangExtract. Keep in mind that these packages are needed as LangExtract requires the llm (when from HF, or Llama-2) in an api-like structure (here port: 8001)
See usage examples, https://github.com/google/langextract?utm_source=chatgpt.com

### Basic usage test of LangExtract
Example taken from langExtract github 
set api key for Gemini in `.env`

In [ ]:
# 1. Define the prompt and extraction rules
prompt = textwrap.dedent(
    """\
    Extract characters, emotions, and relationships in order of appearance.
    Use exact text for extractions. Do not paraphrase or overlap entities.
    Provide meaningful attributes for each entity to add context."""
)

# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="ROMEO. But soft! What light through yonder window breaks? It is the east, and Juliet is the sun.",
        extractions=[
            lx.data.Extraction(
                extraction_class="character",
                extraction_text="ROMEO",
                attributes={"emotional_state": "wonder"},
            ),
            lx.data.Extraction(
                extraction_class="emotion",
                extraction_text="But soft!",
                attributes={"feeling": "gentle awe"},
            ),
            lx.data.Extraction(
                extraction_class="relationship",
                extraction_text="Juliet is the sun",
                attributes={"type": "metaphor"},
            ),
        ],
    )
]

In [ ]:
# The input text to be processed
input_text = "Lady Juliet gazed longingly at the stars, her heart aching for Romeo"

# Run the extraction
result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash",
)

In [ ]:
result

In [ ]:
lx.io.save_annotated_documents(
    [result], output_name="extraction_results.jsonl", output_dir="."
)

# Generate the visualization from the file
html_content = lx.visualize("extraction_results.jsonl")
with open("visualization.html", "w") as f:
    if hasattr(html_content, "data"):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

### Prompt and few-shot examples

In [ ]:
# 1. Define the prompt and extraction rules
prompt = textwrap.dedent(
    """
    Extract from the context information about the flood-affected infrastructure_type, its damage, its geolocation, as well as about 
    the societal or economic impacts which resulted from the infrastructure failure.
    However, if you do not find any information about societal or economic impacts, then return for these fields a "NAN" value.

    Use exact text for extractions. DO NOT paraphrase or overlap entities.
    Provide meaningful attributes for each entity to add context.

 """
)

# 2. Provide some high-quality examples to guide the model
few_shot_examples = [
    lx.data.ExampleData(
        text="More than 130 km of motorways were closed directly after the event, of which 50 km were still closed two months later, with an estimated repair cost of EUR100 million (Hauser, 2021). ",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="motorways",
                attributes={"time": "directly after the event"},
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR100 million",
                attributes={"type": "repair cost"},
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Of the 112 bridges in the flooded 40 km of the Ahr valley (Rhineland-Palatinate), 62 bridges were destroyed, 13 were severely damaged and only 35 were in operation a month after the flood event (MDR, 2021).",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="bridges",
                attributes={"damage type": "destroyed", "number": "62"},
            ),
            lx.data.Extraction(
                extraction_class="geolocation",
                extraction_text="Ahr valley",
                attributes={"region": "Rhineland-Palatinate"},
            ),
        ],
    ),
    lx.data.ExampleData(
        text=" In total, at least 220 casualties have been reported, with insured loss estimates of approximately EUR 150 million–EUR 250 million in the Netherlands (Verbond voor Verzekeraars, 2022), "
        "EUR 2.2 billion in Belgium (Assuralia, 2022) and EUR 8.2 billion (GDV, 2022) in Germany. "
        "The event caused major damages to residential and commercial structures and to many critical infrastructure (CI) assets. ",
        extractions=[
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 150 million–EUR 250 million",
                attributes={
                    "type": "insured loss estimates",
                    "geolocation": "Netherlands",
                    "citation": "Verbond voor Verzekeraars, 2022",
                },
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 2.2 billion",
                attributes={
                    "type": "insured loss estimates",
                    "geolocation": "Belgium",
                    "citation": "Assuralia, 2022",
                },
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 8.2 billion",
                attributes={
                    "type": "insured loss estimates",
                    "geolocation": "Germany",
                    "citation": "GDV, 2022",
                },
            ),
        ],
    ),
]

### Appy LangExtract

In [15]:
context_example = "In mid-July 2021, a persistent low-pressure system caused extreme precipitation in parts of the Belgian, German and Dutch catchments of the Meuse and Rhine rivers. This led to record-breaking water levels and severe ﬂooding (Mohr et al., 2022). Comparable heavy precipitation events in this area have never been registered in most of the affected areas before (Kreienkamp et al., 2021). The German states most affected include Rhineland-Palatinate (Rheinland-Pfalz), with damage to the Ahr River valley (Ahrtal), several regions in\n\nthe Eiffel National Park, to the city of Trier. Flooding in Belgium was concentrated in the Vesdre River valley (districts of Pepinster, Ensival and Verviers), the Meuse River valley (Maaseik, Liége), the Gete River valley (Herk-de-Stad and Halen) and southeast Brussels (Wavre). The Netherlands experienced ﬂooding, mostly concentrated in the southern district of Limburg. In total, at least 220 casualties have been reported, with insured loss estimates of approximately EUR 150 million–EUR 250 million in the Netherlands (Verbond voor Verzekeraars, 2022), ∼ EUR 2.2 billion in Belgium (Assuralia, 2022) and ∼ EUR 8.2 billion (GDV, 2022) in Germany. The event caused major damages to residential and commercial structures and to many critical infrastructure (CI) assets. Not only vital functions for ﬁrst responders were affected (e.g. hospitals, ﬁre departments), but also railways, bridges and utility networks (e.g. water and electricity supply) were severely damaged, expecting to take months to years to fully rebuild.\n\nCI is often considered to be the backbone of a well-functioning society (Hall et al., 2016), which is particularly eminent during natural hazards and disasters. For instance, failure of electricity or telecommunication services immediately causes disruptions in the day-to-day functioning of people and businesses, including those outside the directly affected area. Despite the (academic) agreement that failure of infrastructure systems may cause (large-scale) societal disruptions (Garschagen and Sandholz, 2018; Hallegatte et al., 2019; Fekete and Sandholz, 2021), empirical evidence on the impacts of extreme weather events on these systems is still\n\nPublished by Copernicus Publications on behalf of the European Geosciences Union.\n\nE. E. Koks et al.: Flood impacts to infrastructure\n\nlimited. This brief communication provides an overview of the observed ﬂood impacts to large-scale infrastructure systems during the 2021 mid-July western European ﬂood event and how reconstruction of these large-scale systems has progressed. Next, we highlight how some of these observations compare to academic modelling approaches. We conclude with suggestions on moving forward in CI risk modelling, based on the lessons learned from this extreme event.\n\nIn Germany, road and railway infrastructure was severely damaged as documented exemplarily in Fig. 1. Cost estimates reach up to EURO 2 billion Euro (MDR, 2021). More than 130 km of motorways were closed directly after the event, of which 50 km were still closed two months later, with an estimated repair cost of EUR 100 million (Hauser, 2021). Of the 112 bridges in the ﬂooded 40 km of the Ahr valley (Rhineland-Palatinate), 62 bridges were destroyed, 13 were severely damaged and only 35 were in operation a month after the ﬂood event (MDR, 2021). Over 74 km of roads, paths and bridges in the Ahr valley have been (critically) damaged. In some cases, repairs are expected to take months to years (Zeit Online, 2021). For example, major freeway sections, including parts of the A1 motorway, were closed until early 2022 (24Rhein, 2022). In addition, about 50 000 cars were damaged, causing insurance claims of some EUR 450 million (ADAC, 2021). The German railway provider Deutsche Bahn expects asset damages of around EUR 1.3 billion. Among other things, 180 level crossings, almost 40 signal boxes, over 1000 catenary and signal masts, and 600 km of tracks were destroyed, as well as energy supply systems, elevators and lighting systems (MDR, 2021). As of 11 April 2022, 14 of the affected rail stretches are fully functional again. The less damaged stretches were functional again within 3 months, while some of the most damaged sections in the Ahr valley are expected to be ﬁnished by the end of 2025 (DB, 2022). In Belgium, approximately 10 km of railway tracks and 3000 sleeper tracks have to be replaced; 50 km of catenary needs to be repaired; and 70 000 t of railway track bed needs to be placed, with estimated costs between EUR 30 million–EUR 50 million (Rozendaal, 2021a). Most damages have been repaired within 2 weeks. The most severely damaged railway line (between the villages of Spa and Pepinster) was reopened again on 3 October 2021 (Rozendaal, 2021b). In the Netherlands, no large-scale damage has been reported to transport infrastructure. A few national highways were partly ﬂooded (e.g. the A76 in both directions) or brieﬂy closed (&lt; 3 d) because of the potential of ﬂooding. Most likely due to relative low-ﬂow velocities, damage to Dutch national road infrastructure was limited. Several railway sections were closed (e.g. the rail-\n\nway section between Maastricht and Liége) and some damage occurred to the railway infrastructure, in particular to the electronic “track circuit” devices and saturated railway embankments (Prorail, 2021).\n\nAt the peak of the event, around 200 000 people experienced power outages in Germany. Electricity infrastructure was severely damaged in North Rhine-Westphalia and Rhineland-Palatinate. However, within 2 d around 50 % of the power was restored through repairs and temporary ﬁxes. Within 8 weeks, no emergency power generators were required anymore, with most of the power infrastructure restored in Germany’s affected areas. Some areas, however, only had permanent power infrastructure after 6 months (Westnetz, 2022). The gas distribution network in the Ahr valley was severely damaged. Approximately 133 km of natural gas pipelines, 8500 gas metres, 3400 house pressure regulators, 7220 of the approximately 8000 household connections, and 31 systems measuring and regulating gas pressure have been damaged or destroyed (SWR, 2021). Gas supply was almost fully restored within 4.5 months after the ﬂood event (Energienetze Mittelrhein, 2021). In Belgium, approximately 41 500 people experienced power outages at the peak of the event. This was the result of both damaged and deliberately switched-off electrical cabinets to prevent serious damages. It took around 3 weeks to fully restore power. Similar to Germany, severe damage had been observed to the gas network. In the villages around Liége, such as Chaudfontaine and Pepinster (Belgium), gas supply was fully recovered within 5 months (Grosjean, 2021; De Wolf, 2021). In the Netherlands, 1000– 2000 households experienced a loss of electricity supply at the peak of the event. Between 100 to 200 households had no gas supply. Within several days, electricity supply was restored (Task Force Fact Finding Hoogwater, 2021).\n\nIn the region of Rhineland-Palatinate (Germany), most drinking water supply was restored within 2 months (Hochwasser Ahr, 2021a). However, sewage treatment plants in Altenahr, Mayschoss and Sinzig had been largely destroyed (Hochwasser Ahr, 2021b), and it is expected to take at least 1.5 years to fully repair most sewage treatment plants. Emergency sewage treatment plants have been built in the meantime (GA, 2021). In the Erft region 7 out of 31 wastewater facilities had been destroyed. Many facilities reported pollution of oil and diesel, forming layers up to 15 cm thick (Kuhn, 2021). In addition, much of the groundwater (and soil) in the ﬂood region was mixed with oil (from destroyed residential oil tanks), chemicals such as fertilizers (from wineries and other agriculture) and chemicals from nearby industrial plants. In Sinzig, 3.6 × 106 L of oil–water mixture was recycled, gaining 3600 m3 of oil, to be reused for heating and\n\nNat. Hazards Earth Syst. Sci., 22, 3831–3838, 2022\n\nE. E. Koks et al.: Flood impacts to infrastructure\n\nFigure 1. Damage in the Ahr valley, Germany (images taken on 11 August 2021). (a) Destruction of federal highway B266 (A1) and railway (A2) near Heimersheim. (b) Further upstream in the Ahr valley (Altenburg), large stretches of the Ahrtalbahn railway have been destroyed (B1) and the few remaining road and rail bridges show signs of temporary repairs (B2). (c) Riverbed erosion uncovered and destroyed many cables supposed to lie more than 80 cm below surface level (C1) as well as sewers (C2). (d) Inundated electricity distribution infrastructure (D1), road erosion and stabilization (D2), uncovered cables (D3), and collapsed buildings in Schuld. Pictures by Margreet van Marle/Deltares/GEER-association, distributed under Creative Commons Attribution 4.0 license.\n\nindustrial usage (Kuhn, 2021). In the heavily destroyed town of Bad Münstereifel (in the state of North Rhine-Westphalia), drinking water supply was re-established within 5 d after the ﬂood event (most frequently through emergency tanks), and about 50 % of the city centre was reconnected to the freshwater network shortly thereafter however, water had to be boiled before consumption until about 1 month later (Bad Münstereifel, 2021). In Belgium, several towns experienced disruptions in water supply (in particular as a result of pollution). Directly after the event, approximately 3400 families had no access to potable water. Within less than a week, this was reduced to around 1650 families (Terzake, 2021). It took, however, 6 months to rebuild the permanent water supply infrastructure (SWDE, 2022). In the Netherlands, little to no problems have been recorded with regards to water supply.\n\nWe found no information regarding direct impact on solid-waste facilities as a result of the ﬂood event. However, there is a large pressure on the solid-waste sector to clean the affected areas; 1 month after the event, we observed dozens of large temporary waste ﬁlls and frequent incidences of oil pollution in Rhineland-Palatinate during a ﬁeld visit. In the Ahrweiler district alone, the ﬂood caused as much solid waste as normally would be collected over 30 years. In Belgium, the amount of solid waste is estimated around 160 000 t, stored at several places, such as the abandoned highway track A601. This highway has been used for approximately 9 months as a temporary storage for debris (Couplez, 2022). In the Netherlands, there have been primarily problems with waste deposits along the river banks, which is mostly the solid waste transported by the river from further upstream. Thousands of tonnes of tree debris (logs and\n\nNat. Hazards Earth Syst. Sci., 22, 3831–3838, 2022\n\nE. E. Koks et al.: Flood impacts to infrastructure\n\nof running water and electricity (Ärzte Zeitung, 2021). After 1.5 months, medical care was guaranteed again in the most affected regions in Rhineland-Palatinate (Hochwasser Ahr, 2021c). In the state of North Rhine-Westphalia, approximately 68 hospitals have been affected, of which several have been affected severely and will take at least 1.5 years to be rebuilt (Fig. 2). Direct damages are estimated to be at least EUR 100 million to repair all medical facilities (Korzilius, 2021). In the town of Eschweiler (Germany), for example, the basement of the hospital was ﬂooded, as well as the outbuildings and the entire outdoor area. The power supply collapsed, the entire building technology was destroyed and some 300 patients had to be evacuated by helicopter. Property damage is expected to be around EUR 50 million. Within 3.5 weeks, the hospital was partly operational, and within 3 months, all hospital operations continued normally (SAH Eschweiler, 2021). The Mutterhaus Ehrang hospital in Trier (Germany) is now permanently closed as the hospital is too severely damaged to rebuild. Furthermore, in the region of Rhineland-Palatinate (Germany), 19 daycare centres and 17 schools suffered damage from the ﬂoods, affecting more than 8000 students (Staib, 2021). Approximately 4 months after the ﬂood event, the district of Bad Neuenahr-Ahrweiler established emergency educational facilities using 297 containers that serve as classrooms, ofﬁces and dining facilities for more than 800 students (Wiesbadener Kurier, 2021). In Belgium, various rural clinics have been affected and were unable to provide any services. Concurrently, in the most affected areas, general-practitioner facilities have been completely destroyed (Le Spécialiste, 2021). In the Netherlands, one nursing home was ﬂooded, and one hospital was evacuated as a precautionary measure.\n\nMost often, large-scale object-based infrastructure impact studies (e.g. Bubeck et al., 2019) only disclose aggregated risk metrics (i.e. country-level risk estimates), which hampers veriﬁcation and validation with observed impacts on smaller scales."

In [ ]:
PARSED_TEXT_DIR = "../" + s.PATH_DATA + "parsed_documents/"


filename = "Koks et al 2022 Brief communication_cleaned.md"
filepath = Path(PARSED_TEXT_DIR, filename)
filename_stem = filepath.stem
modelname = "gemini-2.5-flash"


print(
    f"\n\n ######## -------- Processing document: {filepath.name} -------- ######## \n"
)

## extract authors, publication year and title
citation_pattern = r"(.*?)(\d{4})(.*)"  # split at first occurrence of year
try:
    authors, year, title = re.findall(citation_pattern, filename_stem)[0]
    citation = f"{authors} {year}"
except AttributeError as e:
    print(f"Could not extract citation from title: {e}")
    citation = filename_stem


## load doc
# with open(filepath, "r") as file:
#     content = file.read()
# doc = [lx.data.Document(content)]  # wrap content in Document object


print(
    f"\n  #############  -------- Text-2-Data: {filepath.name} -------- #############  \n"
)


response = lx.extract(
    text_or_documents=context_example, #doc[0].text,
    prompt_description=prompt,
    examples=few_shot_examples,
    model_id=modelname,  # "gemini-2.5-pro",
    extraction_passes=1,  # decrease recall but process faster
    max_workers=4,
    max_char_buffer=1000      # Break big text into small pieces
)
# response = lx.extract(
#     text_or_documents=doc[0].text,
#     prompt_description=prompt,
#     examples=few_shot_examples,
#     model_id="gpt-4o-mini",
#     fence_output=True,              # Required for OpenAI
#     use_schema_constraints=False    # OpenAI doesn't support constraints
# )
print(response)

lx.io.save_annotated_documents(
    [response],
    output_name=f"{modelname.replace('.', '')}_{filename_stem}.jsonl",
    output_dir=s.PATH_DATA + "/langextract_output/",
)



 ######## -------- Processing document: Koks et al 2022 Brief communication_cleaned.md -------- ######## 


  #############  -------- Text-2-Data: Koks et al 2022 Brief communication_cleaned.md -------- #############  



LangExtract: model=gemini-2.5-flash [00:00]

LangExtract: model=gemini-2.5-flash, current=9,420 chars, processed=0 chars:  [00:00]2025-12-08 13:19:36,091 - INFO - AFC is enabled with max remote calls: 10.
2025-12-08 13:19:36,095 - INFO - AFC is enabled with max remote calls: 10.
2025-12-08 13:19:36,096 - INFO - AFC is enabled with max remote calls: 10.
2025-12-08 13:19:36,099 - INFO - AFC is enabled with max remote calls: 10.
2025-12-08 13:19:41,902 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2025-12-08 13:19:41,906 - INFO - AFC is enabled with max remote calls: 10.
2025-12-08 13:19:42,312 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2025-12-08 13:19:42,316 - INFO - AFC is enabled with max remote calls: 10.
2025-12-08 13:19:45,398 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/

AnnotatedDocument(extractions=[Extraction(extraction_class='infrastructure_type', extraction_text='NAN', char_interval=None, alignment_status=None, extraction_index=1, group_index=0, description=None, attributes=None), Extraction(extraction_class='economic_impact', extraction_text='NAN', char_interval=None, alignment_status=None, extraction_index=2, group_index=0, description=None, attributes=None), Extraction(extraction_class='location', extraction_text='Ahr River valley (Ahrtal)', char_interval=CharInterval(start_pos=486, end_pos=511), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=3, group_index=0, description=None, attributes={'region': 'Rhineland-Palatinate (Rheinland-Pfalz)'}), Extraction(extraction_class='infrastructure_type', extraction_text='NAN', char_interval=None, alignment_status=None, extraction_index=4, group_index=1, description=None, attributes=None), Extraction(extraction_class='economic_impact', extraction_text='NAN', char_interval=No

LangExtract: Saving to langextract_Koks et al 2022 Brief communication_cleaned.jsonl: 1 docs [00:00, 297.53 docs/s]

✓ Saved 1 documents to langextract_Koks et al 2022 Brief communication_cleaned.jsonl


In [17]:
# Display the response from the previous cell's extraction
response


AnnotatedDocument(extractions=[Extraction(extraction_class='infrastructure_type', extraction_text='NAN', char_interval=None, alignment_status=None, extraction_index=1, group_index=0, description=None, attributes=None), Extraction(extraction_class='economic_impact', extraction_text='NAN', char_interval=None, alignment_status=None, extraction_index=2, group_index=0, description=None, attributes=None), Extraction(extraction_class='location', extraction_text='Ahr River valley (Ahrtal)', char_interval=CharInterval(start_pos=486, end_pos=511), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=3, group_index=0, description=None, attributes={'region': 'Rhineland-Palatinate (Rheinland-Pfalz)'}), Extraction(extraction_class='infrastructure_type', extraction_text='NAN', char_interval=None, alignment_status=None, extraction_index=4, group_index=1, description=None, attributes=None), Extraction(extraction_class='economic_impact', extraction_text='NAN', char_interval=No

In [28]:
# Look at all extracted entities
for i in response.extractions:
   print(f"Type: {i.extraction_class}")
   print(f"Text: '{i.extraction_text}'")
   try:
        print(f"Location: chars {i.char_interval.start_pos}-{i.char_interval.end_pos}")
   except AttributeError:
        print("Location: N/A")
   print(f"Attributes: {i.attributes}")
   print("---")

Type: infrastructure_type
Text: 'NAN'
Location: N/A
Attributes: None
---
Type: economic_impact
Text: 'NAN'
Location: N/A
Attributes: None
---
Type: location
Text: 'Ahr River valley (Ahrtal)'
Location: chars 486-511
Attributes: {'region': 'Rhineland-Palatinate (Rheinland-Pfalz)'}
---
Type: infrastructure_type
Text: 'NAN'
Location: N/A
Attributes: None
---
Type: economic_impact
Text: 'NAN'
Location: N/A
Attributes: None
---
Type: location
Text: 'Eiffel National Park'
Location: chars 537-557
Attributes: {'region': 'German states'}
---
Type: infrastructure_type
Text: 'NAN'
Location: N/A
Attributes: None
---
Type: economic_impact
Text: 'NAN'
Location: N/A
Attributes: None
---
Type: location
Text: 'city of Trier'
Location: chars 566-579
Attributes: {'region': 'German states'}
---
Type: infrastructure_type
Text: 'NAN'
Location: N/A
Attributes: None
---
Type: economic_impact
Text: 'NAN'
Location: N/A
Attributes: None
---
Type: location
Text: 'Vesdre River valley (districts of Pepinster, Ensiva

In [ ]:

# lx.io.save_annotated_documents([result], output_name="../../data/llm_outputs/extraction_results.jsonl", output_dir=".")

# Generate the visualization from the file
html_content = lx.visualize("../" + s.PATH_DATA + "/langextract_output/" + f"{modelname.replace('.', '')}_langextract_{filename_stem}.jsonl")
with open("../" + s.PATH_DATA + "/langextract_output/""visualization.html", "w") as f:
    if hasattr(html_content, 'data'):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

LangExtract: Loading langextract_Koks et al 2022 Brief communication_cleaned.jsonl: 100%|██████████| 71.9k/71.9k [00:00<00:00, 66.1MB/s]

✓ Loaded 1 documents from langextract_Koks et al 2022 Brief communication_cleaned.jsonl


In [ ]:
?lx.data.Document

In [ ]:
doc[0].tokenized_text.__dir__()

In [ ]:
import gc
import torch

# clean up after each document
gc.collect()
torch.cuda.empty_cache()

### Testing vLLM and FastChat

In [ ]:
# # install before langextract and vllm

# # --tensor-parallel-size 1 --gpu-memory-utilization 0.90
# !python -m vllm.entrypoints.openai.api_server \
#     --model meta-llama/Llama-2-7b-chat-hf \
#     --dtype auto \
#     --port 8000 \
#     --gpu-memory-utilization 0.90


## test with fastchat loading
# documentation: https://fastchat.mintlify.app/install
# uv add fschat[model_worker,webui]

import os


login(token=os.environ["HUGGINGFACE_TOKEN"])  # TODO replace by using pydantic settings

## Import NOTE (cpu/cuda/OOM):
## for 7b we need around 20GB VRAM
# first try to run in with some optimazation (--load-8bit) and to run with CUDA --> make sure to compile against local cuda by running in terminal (adapt cuda version):
# CUDACXX=/usr/local/cuda-13/bin/nvcc CMAKE_ARGS="-DLLAMA_CUBLAS=on -DCMAKE_CUDA_ARCHITECTURES=native" FORCE_CMAKE=1 pip install .
# if GPU still results in OOM --> pin down to cpu only "--device cpu"
# start worker at port 8001, NOTE: set specific port only when default port s already used by other process (eg. docker container - vector db)
# !python3 -m fastchat.serve.controller --port 8004 & !python3 -m fastchat.serve.model_worker --model-path meta-llama/Llama-2-7b-chat-hf  --load-8bit --device cpu --port 8004

# start controller and model worker in temrinal
!python3 -m fastchat.serve.controller --port 8001  & python3 -m fastchat.serve.model_worker --model-path meta-llama/Llama-2-7b-chat-hf --device cpu --port 8005


# !python3 -m fastchat.serve.cli --model-path meta-llama/Llama-2-7b-chat-hf  --load-8bit

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ["HUGGINGFACE_TOKEN"])

#### response

In [ ]:
safety_df = df_responses.copy()

# save to disk along with prompt text


OUTPUT_DIR = "../" + s.PATH_DATA + "llm_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

outfile_name = "ci_failure_impacts_responses_llama2_with_eco_explicit_3.csv"
outfile_path = OUTPUT_DIR + outfile_name


if not os.path.isfile(outfile_path):
    outfile_response_path = Path(outfile_path)
    outfile_prompt_path = Path(
        OUTPUT_DIR + "prompt_" + outfile_name.replace(".csv", ".txt")
    )

    print(f"Saving prompt and LLm response to {outfile_response_path.parent} ...")
    with open(outfile_prompt_path, "w") as f:
        f.write(prompt_template)
    safety_df.to_csv(outfile_response_path, index=False)
else:
    print(
        f"Output file {Path(outfile_name).stem} already exists. Skip saving to avoid overwriting ..."
    )

In [ ]:
## 10 min


df_ci_geo

##  Test Llama 3 loaded with Ollama and applied in LangExtract

Note. decided for llama instead of Mistral or GPT-J, as the former is probably quite similar in its performance (and easy to setup in Ollama) and the later is too much outdated compared to Llama 3


Langextract does not support Meta models (eg llama) directly only via Ollama, thus we need an API key first to start our Ollama server :)

Steps to do to when we want to run Ollamas llama/mistral model inside a docker container. Benfits are it makes the installation independent of the local machine + later facilitates distributability of our software
* Setup ollama for docker + GPU setings by installing [nvidia container toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html#installation ) which we need to run Ollama model with my GPU inside a docker container
* then configure docker to use nvidita driver via using nvidia toolkit - if possible in rootless mode which means your containers and docker daemon can be run without root user privileges (`nvidia-ctk runtime configure --runtime=docker --config=$HOME/.config/docker/daemon.json`). Then restart your docker daemon (for detailed documentation, see [here](https://medium.com/cyberark-engineering/how-to-run-llms-locally-with-ollama-cb00fa55d5de ) and [here](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html#configuration)): 
```
nvidia-ctk runtime configure --runtime=docker --config=$HOME/.config/docker/daemon.json
systemctl --user restart docker
sudo nvidia-ctk config --set nvidia-container-cli.no-cgroups --in-place
```  

* start the container: `docker run -d --gpus=all -v ollama:/root/.ollama -p 11434:11434 --name ollama ollama/ollama`, if needed with superuser rights. Note. the first ollama refers to the container name, the second to the client name- See also [here](https://hub.docker.com/r/ollama/ollama)
* check if the ollama server is running on REST API, by checking message for `http://localhost:11434/` in your web browser
* Then pull the model you want to use , e.g `ollama pull llama3` and the `ollama serve` (see, [here](https://github.com/google/langextract?tab=readme-ov-file#api-key-setup-for-cloud-models)), When you use ollama in a container use: `docker exec -it <containername> ollama pull llama3` , then `docker exec -it <containername> ollama serve`  . In case you get the info that the server port(here 11434) is already in use, check the processes on this port (`sudo lsof -i :11434`) and kill it, maybe you also need to stop ollama server: `systemctl stop ollama` and/or reload the daemon (see similar issue discussed [here](https://github.com/ollama/ollama/issues/707))
* In case a container with an image already exists: simply run `docker compose up -d`

* Lastly, implement pulled model in LangExtract framework:
```
# test with local model from Ollama
result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3",  # Automatically selects Ollama provider
    model_url="http://localhost:11434",  # where our Ollama server is running 
    fence_output=False,
    use_schema_constraints=False
)
```

Note:
* In the examples above, the model runs on localhost, port: 11434
* Optional: pull model and run in terminal, e.g. llama-3, in container named ollama. `docker exec -it ollama ollama run llama3`
* Get your Ollama key [here](https://signin.ollama.com/?client_id=client_01JX0QMHD43PFFCCNXH82A6K8B&redirect_uri=https%3A%2F%2Follama.com%2Fauth%2Fcallback&authorization_session_id=01KC98BGZP7TJNPGYZE1P27SC6)
* check which models are available in Ollama. `$ curl https://ollama.com/api/tags` or via [Ollama documentation](https://ollama.com/library/)

For more info see also: 
* Batch processing and long texts or about functioning of langExtract: [Weights & Biases ](https://wandb.ai/wandb_fc/genai-research/reports/LangExtract-Transform-text-into-structured-data-with-AI--VmlldzoxNDI1OTMyNw#:~:text=LangExtract%20is%20open%2Dsource%20and,without%20requiring%20any%20fine%2Dtuning)


### Prompt and few-shot examples


In [ ]:
# 1. Define the prompt and extraction rules
prompt = textwrap.dedent(
    """
    Extract information from the context about the affected infrastructure_type, its damage, its geolocation, as well as about cascading impacts to other infrastructure assets.
    Extract also information about the societal or economic impacts which resulted from the disrupted infrastructure.

    Use the exact text for extractions. DO NOT paraphrase or overlap entities.
    Provide meaningful attributes for each entity to add context.

    Provide in the field "damage" the type of damage to the specific infrastructure.
    If no information about the damage type is found, then return for this field a "NAN" value.
        
    Provide in the field "impacts_to_other_infrastructure_assets" information about cascading impacts to other infrastructure assets mentioned in the context.
    If no information about cascading impacts to other infrastructure assets is found, then return for this field a "NAN" value.
    
    Provide in the field "societal_impact" information about societal consequences of the infrastructure failures mentioned in the context.
    If no information about societal consequences is found, then return for this field a "NAN" value.
    
    Provide in the field "economic_impact" information about economic consequences of the infrastructure failures mentioned in the context.
    If no information about economic consequences is found, then return for this field a "NAN" value.
    
    Finally, evaluate and improve your answer.
    
    """
)

# 2. Provide some high-quality examples to guide the model
few_shot_examples = [

    ## Skounding 2023
    lx.data.ExampleData(
        text="""Elsewhere, the Mediterranean country has been battered by severe storms. On overnight storm in Milan on Monday tore off
        roofs and uprooted trees, blocking roads and disrupting overground transportation in Italy’s financial capital.""",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="roads",
                attributes={"damage": "blocked", "location": "Milan (city)"},
            ),
            lx.data.Extraction(
                extraction_class="impacts_to_other_infrastructure_assets",
                extraction_text="transportation",
                attributes={"damage": "disrupted", "location": "Milan (city)"},
            ),
        ],
    ),
    ## Ferlita 2023
    lx.data.ExampleData(
        text="""Un altro incendio si è verificato a Pioppo,frazione di Monreale nel Palermitano, dove un incendio ha minacciato diverse
        abitazioni in località Casaboli e distrutto gran parte della vegetazione di quell’area. Anche il territorio di Partinico non è
        stato risparmiato: l’incendio è divampato alcuni giorni fa sulla statale 113.""",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="highway",
                attributes={"damage": "affected", "location": "Partinico area", "name": "State highway 113"},
            ),
        ],
    ),
    lx.data.ExampleData(
        text="""Ulteriori danni si sono verificati nei giorni scorsi all’aeroporto di Catania considerato il quinto per ordine d’importanza in
        tutta Italia. A seguito di un incendio divampato all’interno dei terminal, le aree di accesso sono state prontamente chiuse ai
        viaggiatori creando un enorme danno all’economia del territorio, al settore turistico e a diversi professionisti. In termini
        monetari, i danni sono enormi: l’Ente nazionale aviazione civile quantifica un investimento complessivo per impianti e
        manutenzioni di circa 200 mila euro. Il Mec (Movimento elettori consumatori), invece, stima una spesa di circa 40 milioni
        di euro al giorno dovuta alla chiusura dello scalo a causa dell’incendio. Il totale dei danni, ammonterebbe a più di 80
        milioni di euro. L’incendio è avvenuto la scorsa domenica notte presso l’aeroporto Vincenzo Bellini di Catania e, dopo due
        giorni di incessante lavoro per lo spegnimento del fuoco, dal terminal C sono ripresi arrivi e partenze.""",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="airport terminal",
                attributes={"damage": "damaged", "location": "Catania Airport"},
            ),
            lx.data.Extraction(
                extraction_class="impacts_to_other_infrastructure_assets",
                extraction_text="airport",
                attributes={"damage": "closure"},
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="more than EURO 80 million",
            )
        ],
    ),

    ## EFE 2024
    lx.data.ExampleData(
        text="""The most serious disruptions are on roads in Valencia, with closures on several
        sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it
        with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the 
        towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí,
        Guadassuar, Alzira or Chiva (Valencia), among other municipalities.""",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="road",
                attributes={
                    "damage": "closures", 
                    "location": "Valencia area", 
                    "name": "A-3",
                    # "impacts_to_other_infrastructure_assets": "traffic disrupted"
                },
            ),
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="road",
                attributes={
                    "damage": "closures", 
                    "location": "Valencia area", 
                    "name": "A-7",
                    # "impacts_to_other_infrastructure_assets": "traffic disrupted",
                },
            ),
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="road",
                attributes={
                    "damage": "closures", 
                    "location": "Valencia area", 
                    "name": "AP-7",
                    # "impacts_to_other_infrastructure_assets": "traffic disrupted"
                },
            ),
            lx.data.Extraction(
                extraction_class="impacts_to_other_infrastructure_assets",
                extraction_text="traffic",
                attributes={"damage": "disrupted", "location": "Valencia area"}, 
            ),
        ],
    ),
    ## Containerlift 2024
    lx.data.ExampleData(
        text="Nevertheless, the reopening of Valencia and Sagunto ports for maritime traffic marks a significant step forward for the region’s recovery.",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="Port operability",
                attributes={"damage": "temporarily closed", "location": "Valencia port"},
            ),
            lx.data.Extraction(
                extraction_class="impacts_to_other_infrastructure_assets",
                extraction_text="Port operability",
                attributes={"damage": "temporarily closed", "location": "Sagunto port"},
            )
        ],
    ),
    ## Khazai et al 2013
    lx.data.ExampleData(
        text="Durch einen Deichbruch am 10.06. mussten im Landkreis Stendal Fernverkehrsstrecken der Deutschen Bahn AG gesperrt werden, sodass es zu Zugausfällen und langen Verspätungen kommt.",
        extractions=[   # tricky extraction due to non-English language
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="dam",
                attributes={"damage": "dam failure", "location": "Stendal (Landkreis)"},
            ),
            lx.data.Extraction(
                extraction_class="impacts_to_other_infrastructure_assets",
                extraction_text="railway network",
                attributes={"damage": "closure"},
            ),
            lx.data.Extraction(
                extraction_class="impacts_to_other_infrastructure_assets",
                extraction_text="train service",
                attributes={"damage": "disrupted"},
            ),
            lx.data.Extraction(
                extraction_class="impacts_to_other_infrastructure_assets",
                extraction_text="train service",
                attributes={"damage": "cancellations"},
            )
        ],
    ),
    ## Koks et al., 2022
    lx.data.ExampleData(
        text="In Belgium, several towns experienced disruptions in water supply (in particular as a result of pollution).",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="water network",
                attributes={"damage": "contaminated", "location": "Belgium"},
            ),
            lx.data.Extraction(
                extraction_class="impacts_to_other_infrastructure_assets",
                extraction_text="water supply",
                attributes={"damage": "disrupted"},
            ),
        ],
    ),
    lx.data.ExampleData(
        text="In Belgium, several towns experienced disruptions in water supply (in particular as a result of pollution). Directly after the event, approximately 3400 families had no access to potable water.",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="water network",
                attributes={"damage": "contaminated", "location": "Belgium"},
            ),
            lx.data.Extraction(
                extraction_class="impacts_to_other_infrastructure_assets",
                extraction_text="water supply",
                attributes={"damage": "disrupted"},
            ),
            lx.data.Extraction(
                extraction_class="societal_impact",
                extraction_text="3400 families",
                attributes={"damage": "affected"},
            ),
        ],
    ),
       lx.data.ExampleData(
            text="Within the region of Rhineland-Palatinate, it took 2 weeks to ensure 100 % coverage again through emergency communication masts. Within 1 month, most of the network was restored to pre-disaster service provision. After 5 months, broadband has also been restored in the most affected areas, which started in most areas only after power infrastructure was rebuilt.",
            extractions=[
                lx.data.Extraction(
                    extraction_class="infrastructure_type",
                    extraction_text="power infrastructure",
                    attributes={"damage": "affected", "location": "Rhineland-Palatinate"},  # tricky extraction of location-info
                ),
                lx.data.Extraction(
                    extraction_class="impacts_to_other_infrastructure_assets",
                    extraction_text="broadband",
                    attributes={"location": "most areas only after power infrastructure was rebuilt"},
                ),
            ],
        ),
       lx.data.ExampleData(
        text="Of the 112 bridges in the flooded 40 km of the Ahr valley (Rhineland-Palatinate), 62 bridges were destroyed, 13 were severely damaged and only 35 were in operation a month after the flood event (MDR, 2021).",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="bridges",
                attributes={"damage": "destroyed", "number": "62"},
            ),
            lx.data.Extraction(
                extraction_class="geolocation",
                extraction_text="Ahr valley",
                attributes={"region": "Rhineland-Palatinate"},
            ),
        ],
    ),
   

n Belgium, several towns experienced disruptions in water supply (in particular as a result of pollution). 
Directly after the event, approximately 3400 families had no access to potable water

    lx.data.ExampleData(
        text="More than 130 km of motorways were closed directly after the event, of which 50 km were still closed two months later, with an estimated repair cost of EUR100 million (Hauser, 2021). ",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="motorways",
                attributes={"damage": "closure"},
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR100 million",
                attributes={"type": "repair cost"},
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Of the 112 bridges in the flooded 40 km of the Ahr valley (Rhineland-Palatinate), 62 bridges were destroyed, 13 were severely damaged and only 35 were in operation a month after the flood event (MDR, 2021).",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="bridges",
                attributes={"damage": "destroyed", "number": "62"},
            ),
            lx.data.Extraction(
                extraction_class="geolocation",
                extraction_text="Ahr valley",
                attributes={"region": "Rhineland-Palatinate"},
            ),
        ],
    ),
    lx.data.ExampleData(
        text=" In total, at least 220 casualties have been reported, with insured loss estimates of approximately EUR 150 million–EUR 250 million in the Netherlands (Verbond voor Verzekeraars, 2022), "
        "EUR 2.2 billion in Belgium (Assuralia, 2022) and EUR 8.2 billion (GDV, 2022) in Germany. "
        "The event caused major damages to residential and commercial structures and to many critical infrastructure (CI) assets. ",
        extractions=[
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 150 million–EUR 250 million",
                attributes={
                    "type": "insured loss estimates",
                    "geolocation": "Netherlands",
                    "citation": "Verbond voor Verzekeraars, 2022",
                },
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 2.2 billion",
                attributes={
                    "type": "insured loss estimates",
                    "geolocation": "Belgium",
                    "citation": "Assuralia, 2022",
                },
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 8.2 billion",
                attributes={
                    "type": "insured loss estimates",
                    "geolocation": "Germany",
                    "citation": "GDV, 2022",
                },
            ),
        ],
    ),
]

### Appy LangExtract

In [ ]:
PARSED_TEXT_DIR = "../" + s.PATH_DATA + "parsed_documents/"
OUTPUT_DIR = "../" + s.PATH_DATA + "langextract_output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

filename = "Koks et al 2022 Brief communication_cleaned.md"
FILE_PATH = Path(PARSED_TEXT_DIR, filename)
filename_stem = FILE_PATH.stem

modelname = "llama3"


print(
    f"\n\n ######## -------- Processing document: {FILE_PATH.name} -------- ######## \n"
)

## extract authors, publication year and title
citation_pattern = r"(.*?)(\d{4})(.*)"  # split at first occurrence of year
try:
    authors, year, title = re.findall(citation_pattern, filename_stem)[0]
    citation = f"{authors} {year}"
except AttributeError as e:
    print(f"Could not extract citation from title: {e}")
    citation = filename_stem


# ## load doc
# with open(FILE_PATH, "r") as file:
#     content = file.read()
# doc = [lx.data.Document(content)]  # wrap content in Document object
loader = DoclingLoader(FILE_PATH)  # use chunks from Docling.Loader
doc = loader.load()


print(
    f"\n  #############  -------- Text-2-Data: {FILE_PATH.name} -------- #############  \n"
)

## TODO loop makes use of the chunking in docling, however, the implemented chunking strategy in LX might be better,
## however, then `[lx.data.Document(content)]` object needs to be splitted into smaller chunks (currently entire text is 1 chunk)

## NOTE for multiple documents / long text use batch processing (threading, character buffer, number of model passes on the text)
responses = []

for i, _ in enumerate(doc):
    response = lx.extract(
        text_or_documents=doc[i].page_content, # context_example
        prompt_description=prompt,
        examples=few_shot_examples,
        model_id= modelname,  # Automatically selects Ollama provider
        model_url=os.getenv("OLLAMA_HOST", "http://localhost:11434"),  # make explicit where Ollama server is running 
        # language_model_type=lx.inference.OllamaLanguageModel,
        temperature=0.2,
        extraction_passes=1,  # decrease recall but process faster
        # max_workers=4,   # 4 parallel threads
        max_char_buffer=1000,      # Break big text into small pieces
    )
    responses.append(response)


# response = lx.extract(
#     text_or_documents=context_example,
#     prompt_description=prompt,
#     examples=few_shot_examples,
#     model_id="gemma2:2b",  # Automatically selects Ollama provider
#     model_url="http://localhost:11434",
#     fence_output=False,
#     use_schema_constraints=False
# )

print(responses)

lx.io.save_annotated_documents(
    responses,
    output_name=f"{modelname.replace(':', '_')}_{filename_stem}.jsonl",
    output_dir=OUTPUT_DIR,
)



 ######## -------- Processing document: Koks et al 2022 Brief communication_cleaned.md -------- ######## 



2025-12-18 13:16:05,654 - INFO - Going to convert document batch...
2025-12-18 13:16:05,656 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2025-12-18 13:16:05,656 - INFO - Processing document Koks et al 2022 Brief communication_cleaned.md
2025-12-18 13:16:05,702 - INFO - Finished converting document Koks et al 2022 Brief communication_cleaned.md in 0.05 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (556 > 512). Running this sequence through the model will result in indexing errors



  #############  -------- Text-2-Data: Koks et al 2022 Brief communication_cleaned.md -------- #############  



LangExtract: Processing, current=928 chars, processed=0 chars:  [00:08]
LangExtract: Processing, current=955 chars, processed=0 chars:  [00:03]
LangExtract: Processing, current=531 chars, processed=0 chars:  [00:03]
LangExtract: Processing, current=1,024 chars, processed=0 chars:  [00:23]
LangExtract: Processing, current=1,244 chars, processed=0 chars:  [00:09]
LangExtract: Processing, current=1,095 chars, processed=0 chars:  [00:23]
LangExtract: Processing, current=1,080 chars, processed=0 chars:  [00:15]
LangExtract: Processing, current=392 chars, processed=0 chars:  [00:05]
LangExtract: Processing, current=1,271 chars, processed=0 chars:  [00:09]
LangExtract: Processing, current=407 chars, processed=0 chars:  [00:10]
LangExtract: Processing, current=1,043 chars, processed=0 chars:  [00:22]
LangExtract: Processing, current=769 chars, processed=0 chars:  [00:05]
LangExtract: Processing, current=898 chars, processed=0 chars:  [00:05]
LangExtract: Processing, current=1,035 chars, proces

[AnnotatedDocument(extractions=[Extraction(extraction_class='infrastructure_type', extraction_text='critical infrastructure', char_interval=CharInterval(start_pos=210, end_pos=233), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=1, group_index=0, description=None, attributes={'damage type': 'impacts'}), Extraction(extraction_class='geolocation', extraction_text='western European', char_interval=CharInterval(start_pos=263, end_pos=279), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=2, group_index=1, description=None, attributes={'region': 'Europe'})], text='Nat. Hazards Earth Syst. Sci., 22, 3831–3838, 2022 https://doi.org/10.5194/nhess-22-3831-2022 © Author(s) 2022. This work is distributed under the Creative Commons Attribution 4.0 License.\nBrief communication: Critical infrastructure impacts of the 2021 mid-July western European ﬂood event\nElco E. Koks1,2, Kees C. H. van Ginkel3,1, Margreet J. E. van Marle3, and Ann

LangExtract: Saving to llama3_langextract_Koks et al 2022 Brief communication_cleaned.jsonl: 27 docs [00:00, 5024.23 docs/s]

✓ Saved 27 documents to llama3_langextract_Koks et al 2022 Brief communication_cleaned.jsonl


In [ ]:
# safety_copy = responses


LangExtract: Saving to llama3_Koks et al 2022 Brief communication_cleaned.jsonl: 27 docs [00:00, 6641.23 docs/s]

✓ Saved 27 documents to llama3_Koks et al 2022 Brief communication_cleaned.jsonl


In [16]:
# Display the response from the previous cell's extraction
response.tokenized_text?

response.tokenized_text.tokens.__len__()

responses[1].extractions


[Extraction(extraction_class='infrastructure_type', extraction_text='critical infrastructure', char_interval=CharInterval(start_pos=183, end_pos=206), alignment_status=<AlignmentStatus.MATCH_FUZZY: 'match_fuzzy'>, extraction_index=1, group_index=0, description=None, attributes={'damage type': 'severely damaged or completely destroyed'}),
 Extraction(extraction_class='geolocation', extraction_text='Germany, Belgium and the Netherlands', char_interval=CharInterval(start_pos=10, end_pos=46), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=2, group_index=1, description=None, attributes={})]

Type:        property
String form: <property object at 0x79a8ff21b290>
Docstring:   <no docstring>

In [26]:
# Look at all extracted entities
for  i in range(len(responses)):
     for j in responses[i].extractions:
          print(f"Type: {j.extraction_class}")
          print(f"Text: '{j.extraction_text}'")
          try:
               print(f"Location: chars {j.char_interval.start_pos}-{j.char_interval.end_pos}")
          except AttributeError:
               print("Location: N/A")
          print(f"Attributes: {j.attributes}")
          print(f"Char span: {j.char_interval}, Token span: {j.token_interval}")
          try:
               print(f"Alignment: {j.alignment_status.value}")
          except AttributeError:
               print("Alignment: N/A")
          print("---")

Type: infrastructure_type
Text: 'critical infrastructure'
Location: chars 210-233
Attributes: {'damage type': 'impacts'}
Char span: CharInterval(start_pos=210, end_pos=233), Token span: TokenInterval(start_index=58, end_index=60)
Alignment: match_exact
---
Type: geolocation
Text: 'western European'
Location: chars 263-279
Attributes: {'region': 'Europe'}
Char span: CharInterval(start_pos=263, end_pos=279), Token span: TokenInterval(start_index=67, end_index=69)
Alignment: match_exact
---
Type: infrastructure_type
Text: 'critical infrastructure'
Location: chars 183-206
Attributes: {'damage type': 'severely damaged or completely destroyed'}
Char span: CharInterval(start_pos=183, end_pos=206), Token span: TokenInterval(start_index=32, end_index=34)
Alignment: match_fuzzy
---
Type: geolocation
Text: 'Germany, Belgium and the Netherlands'
Location: chars 10-46
Attributes: {}
Char span: CharInterval(start_pos=10, end_pos=46), Token span: TokenInterval(start_index=2, end_index=8)
Alignment: m

In [ ]:

# lx.io.save_annotated_documents([result], output_name="../../data/llm_outputs/extraction_results.jsonl", output_dir=".")

# Generate the visualization from the file
html_content = lx.visualize(OUTPUT_DIR + f"{modelname.replace(':', '_')}_{filename_stem}.jsonl")
with open(OUTPUT_DIR + "visualization.html", "w") as f:
    if hasattr(html_content, 'data'):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

LangExtract: Loading llama3_Koks et al 2022 Brief communication_cleaned.jsonl: 100%|██████████| 63.8k/63.8k [00:00<00:00, 36.8MB/s]

✓ Loaded 27 documents from llama3_Koks et al 2022 Brief communication_cleaned.jsonl


## Evaluation